# Data Preprocessing

**Project:** Income Prediction: What Determines Who Earns More? (2.4)

**Team:** Anastasia Sidorova and Paola Cancino

**Date:** 3/22/2026
   
### ⚠️ Key Rule: Split FIRST, then preprocess!
Any step that uses statistics (mean, median, IQR) must be fit on **training data only**.

In [2]:
## import libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


## settings for plots
plt.style.use('seaborn-v0_8')
pd.set_option('display.max_columns',None)

print("✓ Libraries loaded!")

✓ Libraries loaded!


## 1. Load Data

In [5]:
!pip install ucimlrepo

from ucimlrepo import fetch_ucirepo 

# fetch dataset 
adult = fetch_ucirepo(id=2) 
  
# data (as pandas dataframes) 
X = adult.data.features 
y = adult.data.targets 

df = pd.concat([X, y], axis=1)
df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [7]:
df.shape

(48842, 15)

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             48842 non-null  int64 
 1   workclass       47879 non-null  object
 2   fnlwgt          48842 non-null  int64 
 3   education       48842 non-null  object
 4   education-num   48842 non-null  int64 
 5   marital-status  48842 non-null  object
 6   occupation      47876 non-null  object
 7   relationship    48842 non-null  object
 8   race            48842 non-null  object
 9   sex             48842 non-null  object
 10  capital-gain    48842 non-null  int64 
 11  capital-loss    48842 non-null  int64 
 12  hours-per-week  48842 non-null  int64 
 13  native-country  48568 non-null  object
 14  income          48842 non-null  object
dtypes: int64(6), object(9)
memory usage: 5.6+ MB


#### Brief Summary
* The dataset has 48,842 rows and 15 columns.
* The income column is our target.
* An issue that we will improve from the previous noteboook is making our goal more obvious and giving a more descriptive analysis. 

## 2. Drop Rows with Missing TARGET

In [18]:
# DROP rows with missing target
df_clean = df.dropna(subset=['income'])

# Verify no missing targets remain
df_clean.shape

(48842, 15)

#### Observations
* There are no rows with missing target, so the rows and columns are the same.

## 3. Simple Cleaning

In [22]:
## remove duplicates
duplicates = df.duplicated().sum()
print(f'\n Duplicate rows found: {duplicates}')


 Duplicate rows found: 48


In [24]:
if duplicates>0:
    df_clean = df.drop_duplicates()
    print(f'removed {duplicates} duplicated rows')
else:
    print('No duplicates removed')

removed 48 duplicated rows


In [26]:
df_clean = df_clean.drop(columns=['fnlwgt','education'])
print(f'Remaining columns: {df_clean.shape[1]}')

Remaining columns: 13


In [28]:
# encode target variable: yes=1, no=0
df_clean['income'] = (df_clean['income'] == '>50K').astype(int)

print('Target encoded:')
print(df_clean['income'].value_counts())

Target encoded:
income
0    37113
1    11681
Name: count, dtype: int64


In [30]:
# remove impossible values
df_clean= df_clean[df_clean['age'] > 0]
df_clean= df_clean[df_clean['hours-per-week'] > 0]
df_clean= df_clean[df_clean['capital-gain'] >= 0]
df_clean= df_clean[df_clean['capital-loss'] >= 0]

In [32]:
# check current columns

print(f'Shape: {df_clean.shape}')
print(f'\nNumerical: {df_clean.select_dtypes(include=["int64","float64"]).columns.tolist()}')
print(f'\nCategorical: {df_clean.select_dtypes(include="object").columns.tolist()}')

Shape: (48794, 13)

Numerical: ['age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']

Categorical: ['workclass', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']


#### What we removed and why?
* The 'fnlwgt' column was removed because this value would not be relevant when making real world predictions.
* The 'education' column was removed because it's a duplicate of the information in 'education-num' where it's represented numerically.

## 4. Train-Test Split

**Now we split.** After this point, all operations using statistics must:
1. Be **fit on training data only**
2. Be **applied to both** train and test sets

In [36]:
## seperate features and target
X = df_clean.drop('income',axis=1)
y = df_clean['income']

In [38]:
X

,age,workclass,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
0,39,State-gov,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States
1,50,Self-emp-not-inc,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States
2,38,Private,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States
3,53,Private,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States
4,28,Private,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba
...,...,...,...,...,...,...,...,...,...,...,...,...
48837,39,Private,13,Divorced,Prof-specialty,Not-in-family,White,Female,0,0,36,United-States
48838,64,NaN,9,Widowed,NaN,Other-relative,Black,Male,0,0,40,United-States
48839,38,Private,13,Married-civ-spouse,Prof-specialty,Husband,White,Male,0,0,50,United-States
48840,44,Private,13,Divorced,Adm-clerical,Own-child,Asian-Pac-Islander,Male,5455,0,40,United-States


In [40]:
y

0        0
1        0
2        0
3        0
4        0
        ..
48837    0
48838    0
48839    0
48840    0
48841    1
Name: income, Length: 48794, dtype: int32

In [42]:
print(f"\nFeatures (X) shape: {X.shape}")
print(f"Target (y) shape: {y.shape}")


Features (X) shape: (48794, 12)
Target (y) shape: (48794,)


In [44]:
## split first!
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(
    X,y,
    test_size=0.2,
    random_state=4950,
     stratify=y ## keeps same class ratio in both sets
)

In [46]:
print(f'Training:{X_train.shape[0]:,} samples')
print(f'Ttest:{X_test.shape[0]:,} samples')

Training:39,035 samples
Ttest:9,759 samples


In [48]:
X_train

,age,workclass,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
34342,46,Private,9,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,40,United-States
42876,49,State-gov,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,0,40,United-States
30579,17,Private,7,Never-married,Handlers-cleaners,Own-child,Amer-Indian-Eskimo,Male,0,0,20,United-States
36071,48,Private,14,Married-civ-spouse,Sales,Husband,White,Male,0,0,50,United-States
10004,25,Private,9,Married-civ-spouse,Adm-clerical,Wife,White,Female,0,0,40,United-States
...,...,...,...,...,...,...,...,...,...,...,...,...
17989,51,Private,9,Divorced,Prof-specialty,Not-in-family,White,Male,0,0,40,United-States
32274,34,Private,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,45,United-States
24538,32,Private,13,Never-married,Protective-serv,Not-in-family,Black,Male,0,0,30,United-States
3213,74,Federal-gov,9,Widowed,Other-service,Not-in-family,White,Male,0,0,17,United-States


In [50]:
X_test

,age,workclass,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
13993,33,Private,9,Divorced,Other-service,Unmarried,White,Female,0,0,40,United-States
37840,39,State-gov,14,Married-civ-spouse,Prof-specialty,Husband,Black,Male,5013,0,56,United-States
24165,19,Private,9,Never-married,Other-service,Own-child,White,Female,0,0,40,El-Salvador
35501,24,Private,13,Never-married,Other-service,Not-in-family,White,Male,0,0,40,United-States
18240,58,Private,14,Never-married,Adm-clerical,Not-in-family,White,Female,0,0,24,United-States
...,...,...,...,...,...,...,...,...,...,...,...,...
11788,31,Private,5,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,43,United-States
41390,35,Local-gov,13,Never-married,Prof-specialty,Not-in-family,White,Female,0,0,55,United-States
729,65,Self-emp-not-inc,7,Married-civ-spouse,Exec-managerial,Husband,White,Male,9386,0,59,?
36498,46,Private,10,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,40,United-States


In [52]:
print('y_train ratio', y_train.value_counts(normalize=True) * 100)
print('y_test ratio', y_test.value_counts(normalize=True) * 100)

y_train ratio income
0    76.059946
1    23.940054
Name: proportion, dtype: float64
y_test ratio income
0    76.063121
1    23.936879
Name: proportion, dtype: float64


#### Observations

* The training set contains 39,035 rows, while the test set contains 9,759 rows.
* The class distribution is nearly identical between the training and test sets. 

## 5. Handle Missing Values

In [56]:
# check missing values
missing_train = X_train.isna().mean() * 100
print("Percent missing values per column in training set:")
print(missing_train[missing_train > 0])

Percent missing values per column in training set:
workclass         1.962341
occupation        1.967465
native-country    0.561035
dtype: float64


#### Method 1: Drop rows with missing ("unknown") values (Simple but loses data)

In [59]:
# drop rows in training set
cols_to_check = ['workclass', 'occupation', 'native-country']

# create a mask for rows without missing values
mask_train = X_train[cols_to_check].notna().all(axis=1)

# apply mask
X_train = X_train[mask_train]
y_train = y_train[mask_train]

In [61]:
mask_test = X_test[cols_to_check].notna().all(axis=1)

X_test = X_test[mask_test]
y_test = y_test[mask_test]

print("Training set missing values:\n", X_train[cols_to_check].isna().sum())
print("Test set missing values:\n", X_test[cols_to_check].isna().sum())

Training set missing values:
 workclass         0
occupation        0
native-country    0
dtype: int64
Test set missing values:
 workclass         0
occupation        0
native-country    0
dtype: int64


#### Missing value method
workclass / 1.96% / Drop row / Random and under 2%

occupation / 1.96% / Drop row / Random and under 2%

native-country / 0.56% / Drop row / Random and under 2%

## 6. Handle Outliers

#### Method 2: Cap outliers (Winsorization)

In [66]:
# After dropping rows with missing values
X_train_clean = X_train.copy()
X_test_clean = X_test.copy()

In [68]:
# Identify numeric columns
numerical_cols = X_train_clean.select_dtypes(include=['int64', 'float64']).columns.tolist()
print(f'Numerical columns ({len(numerical_cols)}): {numerical_cols}')

Numerical columns (5): ['age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']


In [70]:
# Compute outlier percentage for each numeric feature
for col in numerical_cols:
    Q1 = X_train_clean[col].quantile(0.25)
    Q3 = X_train_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    n_outliers = ((X_train_clean[col] < lower_bound) | (X_train_clean[col] > upper_bound)).sum()
    percent = (n_outliers / len(X_train_clean)) * 100
    print(f'{col}: {n_outliers} outliers ({percent:.2f}%)')

age: 268 outliers (0.70%)
education-num: 1392 outliers (3.66%)
capital-gain: 3186 outliers (8.37%)
capital-loss: 1768 outliers (4.64%)
hours-per-week: 10349 outliers (27.19%)


In [72]:
# Specify which numeric columns to cap (Winsorize) based on your analysis
columns_to_cap = ['education-num']  # e.g., ['capital-gain', 'capital-loss']

def cap_outlier(train_data, test_data, columns):
    train_capped = train_data.copy()
    test_capped = test_data.copy()
    
    for col in columns:
        # Calculate IQR from training set only
        Q1 = train_capped[col].quantile(0.25)
        Q3 = train_capped[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        # Apply Winsorization
        train_capped[col] = train_capped[col].clip(lower=lower_bound, upper=upper_bound)
        test_capped[col] = test_capped[col].clip(lower=lower_bound, upper=upper_bound)
        
    return train_capped, test_capped

# Apply capping
X_train_final, X_test_final = cap_outlier(X_train_clean, X_test_clean, columns_to_cap)

# Check shapes
print(f'Shape of X_train_final: {X_train_final.shape}')
print(f'Shape of X_test_final: {X_test_final.shape}')

# Optional: min/max after capping
for col in columns_to_cap:
    print(f'{col} - train min/max: {X_train_final[col].min()}/{X_train_final[col].max()}')
    print(f'{col} - test min/max: {X_test_final[col].min()}/{X_test_final[col].max()}')

Shape of X_train_final: (38064, 12)
Shape of X_test_final: (9509, 12)
education-num - train min/max: 4.5/16.0
education-num - test min/max: 4.5/16.0


In [74]:
y_train.value_counts()

income
0    28834
1     9230
Name: count, dtype: int64

## Outlier Handling Decisions (Domain-Aware)
age / 0.70% outliers / Keep / Real values that provide important model information.

education-num / 3.66% outliers / Cap / Very high or low education years may distort the model.

capital-gain / 8.37% otliers / Keep / I tried capping capital-gain because it's slightly skewed, however capping didn't change anything. 

capital-loss / 4.64% outliers / Keep / I also tried caping capital-loss because massive capital-loss is rare, however it didn't change anything either.

hours-per-week / 27.18% outliers / Keep / Some people work more hours then others. 

## Key Decisions Summary

We started with 48,842 samples 

#### Before Split:
* There are no rows with missing target, so the rows and columns are the same.
* The 'fnlwgt' column was removed because this value would not be relevant when making real world predictions.
* The 'education' column was removed because it's a duplicate of the information in 'education-num' where it's represented numerically.

#### After Split: 
* The training set contains 39,035 rows, while the test set contains 9,759 rows.
* The class distribution is nearly identical between the training and test sets.
* We dropped the rows workclass, occupation, and native country because they were random and under 2% of the training data.
* We capped education-num, because a variety of people could have a wide range of education years that could worsen the model. 